In [1]:
from turkish_tokenizer_wrapper import tokenize_text

tokens, ids = tokenize_text("merhaba dünya")
print(tokens)  # ['merhaba', '<space>', 'dünya']
print(ids)     # [2036, 1, 224]

['merhaba', '<space>', 'dünya']
[2036, 1, 224]


In [2]:
from turkish_tokenizer import turkish_tokenizer as tt

tokens, ids = tt.tokenize("merhaba dünya")
print(tokens)  # ['merhaba', '<space>', 'dünya']
print(ids)     # [2036, 1, 224]

['merhaba', '<space>', 'dünya']
[2036, 1, 224]


In [3]:
vocab_dict = {**tt.bpe_tokens, **tt.suffixes, **tt.roots}

len(vocab_dict), len(tt.roots), len(tt.suffixes), len(tt.bpe_tokens), len(tt.roots) + len(tt.suffixes) + len(tt.bpe_tokens)

(31356, 24194, 405, 10199, 34798)

In [ ]:
from transformers import AutoTokenizer
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", use_fast=True)
llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)
gemma_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt", use_fast=True)

qwen_tokenizer.vocab_size, llama_tokenizer.vocab_size, gemma_tokenizer.vocab_size

(151643, 128000, 262144)

In [90]:
from transformers import AutoModelForCausalLM
llama_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

In [91]:
llama_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [5]:
gemma_tokenizer.save_vocabulary("qwen_tokenizer")

('qwen_tokenizer/tokenizer.model',)

In [6]:
# Ġ and ▁ are representing the space
qwen_tokenizer.vocab["Ġ"], llama_tokenizer.vocab["Ġ"], gemma_tokenizer.vocab["▁"]

(220, 220, 236743)

In [17]:
import json
with open("qwen_tokenizer/turkish_vocab_dict.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=4)

In [19]:
with open("qwen_tokenizer/llama_vocabs.json", "w", encoding="utf-8") as f:
    json.dump(llama_tokenizer.vocab, f, ensure_ascii=False, indent=4)

In [27]:
vocab_list = list(vocab_dict.keys())

In [41]:
vocab_list[58]

'u'

In [48]:
def tr_capitalize(word):
  # i -> İ
  if word[0] == "i":
    return "İ" + word[1:]
  else:
    return word.capitalize()

tr_capitalize(vocab_list[3158])

'İtibar'

In [87]:
from tqdm import tqdm

vocab_to_llama_token = {}
for i in tqdm(range(len(vocab_list)), desc="Mapping vocabulary to Llama tokens"):
  token = vocab_list[i]
  token_ids = llama_tokenizer.encode(token)
  token_ids0 = llama_tokenizer.encode(tr_capitalize(token))
  # if token starts with kok_ ek_ special_
  if token == "<uppercase>":
    vocab_to_llama_token[token] = "128002"
    continue
  if token == "<space>":
    vocab_to_llama_token[token] = "128003"
    continue
  if token == "<newline>":
    vocab_to_llama_token[token] = "128011"
    continue
  if token == "<tab>":
    vocab_to_llama_token[token] = "128012"
    continue
  if token == "<unknown>":
    vocab_to_llama_token[token] = "128013"
    continue
  if token.startswith("kok_") or token.startswith("ek_") or token.startswith("special_"):
    vocab_to_llama_token[token] = "128014"
  elif (len(token_ids)) > (len(token_ids0)):
    token_ids0.remove(128000)
    # add # between numbers and join them as string
    token_ids0 = "#".join(str(x) for x in token_ids0)
    vocab_to_llama_token[token] = token_ids0
  else:
    token_ids.remove(128000)
    # add # between numbers and join them as string
    token_ids = "#".join(str(x) for x in token_ids)
    vocab_to_llama_token[token] = token_ids


Mapping vocabulary to Llama tokens: 100%|██████████| 31356/31356 [00:01<00:00, 27667.14it/s]


In [88]:
import json
with open("qwen_tokenizer/vocab_to_llama_token.json", "w", encoding="utf-8") as f:
    json.dump(vocab_to_llama_token, f, ensure_ascii=False, indent=4)

In [67]:
llama_tokenizer.encode(vocab_list[3158])

[128000, 275, 106954]

In [86]:
llama_tokenizer.encode("<|reserved_special_token_6|>")

[128000, 128014]